# Week 16, MCP Server Lab (no API key)

```text
# Requirements: pip install mcp
```

> No API key needed. Uses the `mcp` Python SDK (`FastMCP`) to expose ZoroLogistics tools as a Model Context Protocol **server**, then connects an in-notebook MCP **client** over stdio to call them and read their schemas.

Learn the MCP roles, **host / client / server** and the tools/resources/prompts primitives, by building the server side and proving the round trip.

## MCP in one paragraph

MCP standardizes how applications give models access to tools, resources, and prompts, one client-server protocol instead of a bespoke integration per tool. You write a **server** with `FastMCP` (decorate three plain Python functions), and any **client** (an IDE, an agent framework, the in-notebook client below) can call them. The tool's **JSON Schema** is the contract the model reads when deciding to call it, the same "description is prompt engineering" idea from Week 14, now a protocol.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root
_root = pathlib.Path.cwd()
while not (_root / "zoro").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))
REPO_ROOT = str(_root)

import json, os, asyncio, tempfile
import numpy as np
from zoro import data

SEED = 42
rng = np.random.default_rng(SEED)

HAS_MCP = False
try:
    from mcp.server.fastmcp import FastMCP
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client
    HAS_MCP = True
    print("mcp SDK imported OK")
except Exception as e:
    print("mcp SDK not available (", type(e).__name__, "); tool schemas will be shown from the manual definitions.")

print("repo root:", REPO_ROOT)

## The ZoroLogistics tools (plain functions first)

Three tools, deterministic and seeded, exactly like Week 14. The manual schemas below mirror what `FastMCP` will derive from the type hints.

In [ ]:
_ship = data.shipments(n=2_000, seed=SEED)
_lanes = data.lanes(seed=11)
_car = data.carriers(seed=7)
_ship_by_id = {row.shipment_id: row for row in _ship.itertuples()}
_lane_by_id = {row.lane_id: row for row in _lanes.itertuples()}
_car_by_id = {row.carrier_id: row for row in _car.itertuples()}
_policies = {d["doc_id"]: d for d in data.policy_docs()}

def track_shipment(shipment_id):
    sid = str(shipment_id).strip().upper()
    row = _ship_by_id.get(sid)
    if row is None:
        return {"error": "shipment " + sid + " not found"}
    lane = _lane_by_id.get(row.lane_id)
    carrier = _car_by_id.get(row.carrier_id)
    return {
        "shipment_id": sid,
        "status": row.status,
        "carrier": carrier.carrier_name if carrier else row.carrier_id,
        "origin": lane.origin if lane else "?",
        "destination": lane.destination if lane else "?",
        "delay_hours": round(float(row.delay_hours), 1),
        "on_time": bool(row.is_on_time),
    }

def list_carriers():
    return [
        {
            "carrier_id": r.carrier_id,
            "name": r.carrier_name,
            "region": r.region,
            "on_time_rate": round(float(r.on_time_rate), 4),
            "fleet_size": int(r.fleet_size),
        }
        for r in _car.itertuples()
    ]

def get_policy(doc_id):
    doc = _policies.get(str(doc_id).strip().upper())
    if doc is None:
        return {"error": "no policy " + str(doc_id)}
    return {"doc_id": doc["doc_id"], "title": doc["title"], "text": doc["text"]}

# The schemas FastMCP exposes (derived from type hints on the decorated functions).
MANUAL_SCHEMAS = {
    "track_shipment": {
        "title": "track_shipment",
        "description": "Look up a shipment's status, route, delay and on-time prediction by id.",
        "type": "object",
        "properties": {"shipment_id": {"type": "string"}},
        "required": ["shipment_id"],
    },
    "list_carriers": {
        "title": "list_carriers",
        "description": "List all carriers with region, on-time rate and fleet size.",
        "type": "object",
        "properties": {},
    },
    "get_policy": {
        "title": "get_policy",
        "description": "Return a ZoroLogistics policy document by id (POL-001..POL-004).",
        "type": "object",
        "properties": {"doc_id": {"type": "string"}},
        "required": ["doc_id"],
    },
}

print("tool functions ready;", len(MANUAL_SCHEMAS), "manual schemas")

## The FastMCP server

Write a self-contained server file to a temp dir and run it as a **stdio subprocess**, the exact pattern MCP Inspector uses. The file injects the repo root so it can `from zoro import data`.

In [ ]:
SERVER_SOURCE = '''import sys, json
sys.path.insert(0, "__REPO_ROOT__")
from zoro import data
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("zoro")

_ship = data.shipments(n=2000, seed=42)
_lanes = data.lanes(seed=11)
_car = data.carriers(seed=7)
_ship_by_id = {row.shipment_id: row for row in _ship.itertuples()}
_lane_by_id = {row.lane_id: row for row in _lanes.itertuples()}
_car_by_id = {row.carrier_id: row for row in _car.itertuples()}
_policies = {d["doc_id"]: d for d in data.policy_docs()}

@mcp.tool()
def track_shipment(shipment_id: str) -> dict:
    "Look up a shipment's status, route, delay and on-time prediction by id."
    sid = shipment_id.strip().upper()
    row = _ship_by_id.get(sid)
    if row is None:
        return {"error": "shipment " + sid + " not found"}
    lane = _lane_by_id.get(row.lane_id)
    carrier = _car_by_id.get(row.carrier_id)
    return {"shipment_id": sid, "status": row.status,
            "carrier": carrier.carrier_name if carrier else row.carrier_id,
            "origin": lane.origin if lane else "?", "destination": lane.destination if lane else "?",
            "delay_hours": round(float(row.delay_hours), 1), "on_time": bool(row.is_on_time)}

@mcp.tool()
def list_carriers() -> list:
    "List all carriers with region, on-time rate and fleet size."
    return [{"carrier_id": r.carrier_id, "name": r.carrier_name, "region": r.region,
             "on_time_rate": round(float(r.on_time_rate), 4), "fleet_size": int(r.fleet_size)}
            for r in _car.itertuples()]

@mcp.tool()
def get_policy(doc_id: str) -> dict:
    "Return a ZoroLogistics policy document by id (POL-001..POL-004)."
    doc = _policies.get(doc_id.strip().upper())
    if doc is None:
        return {"error": "no policy " + doc_id}
    return {"doc_id": doc["doc_id"], "title": doc["title"], "text": doc["text"]}

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''.replace("__REPO_ROOT__", REPO_ROOT)

SERVER_FILE = pathlib.Path(tempfile.gettempdir()) / "zoro_mcp_server.py"
SERVER_FILE.write_text(SERVER_SOURCE)
print("server written to", SERVER_FILE)
print("inspect it with: npx @modelcontextprotocol/inspector python " + str(SERVER_FILE))

## In-notebook MCP client

Connect a client to the stdio subprocess, list the tools, and call all three, the round trip that proves the server works through the protocol.

In [ ]:
def _result_text(res):
    parts = []
    for c in (getattr(res, "content", None) or []):
        txt = getattr(c, "text", None)
        if txt is not None:
            parts.append(txt)
    return "\n".join(parts)

schemas = {}
results = {}
ROUND_TRIP = 0

if HAS_MCP:
    async def _roundtrip():
        server_params = StdioServerParameters(command=sys.executable, args=[str(SERVER_FILE)])
        async with stdio_client(server_params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                schemas.update({t.name: t.inputSchema for t in tools.tools})
                results["track_shipment"] = await session.call_tool("track_shipment", arguments={"shipment_id": "S0000123"})
                results["list_carriers"] = await session.call_tool("list_carriers", arguments={})
                results["get_policy"] = await session.call_tool("get_policy", arguments={"doc_id": "POL-002"})
    asyncio.run(_roundtrip())
    ROUND_TRIP = len([r for r in results.values() if not getattr(r, "isError", False)])
    print("client round trip complete;", ROUND_TRIP, "tools called without error")
else:
    print("(mcp SDK absent, showing manual schemas and direct tool calls instead)")

# Always print the tool results (direct calls when the SDK is absent).
for name, res in results.items():
    print(name, "->", _result_text(res)[:160])
if not results:
    print("track_shipment ->", track_shipment("S0000123"))
    print("list_carriers ->", list_carriers()[:2])
    print("get_policy ->", get_policy("POL-002")["title"])

## The tool schema JSON

The schema is the contract between model and server. Below: the real schema returned by `list_tools()` when the SDK is present, otherwise the manual definition.

In [ ]:
print("tool schema JSON (track_shipment):")
real_schema = schemas.get("track_shipment") or MANUAL_SCHEMAS["track_shipment"]
print(json.dumps(real_schema, indent=2))
print()
print("tool schema JSON (list_carriers):")
print(json.dumps(schemas.get("list_carriers") or MANUAL_SCHEMAS["list_carriers"], indent=2))
print()
print("server exposes", len(schemas) if schemas else len(MANUAL_SCHEMAS), "tools")

In [ ]:
# Final number: how many tools the in-notebook MCP client called through the protocol.
# (0 when the mcp SDK is not installed; 3 when the round trip succeeded.)
print("ROUND_TRIP_TOOLS", ROUND_TRIP)